# Xây dựng một vài chức năng cơ bản cho vnstock

Phần này sẽ xây dựng, một vài chức năng trước khi cho vào UI **vnstock** để truy xuất dữ liệu thị trường chứng khoán Việt Nam:
- Liệt kê các mã cổ phiếu
- Dữ liệu giá lịch sử và thời gian thực
- Thông tin doanh nghiệp và báo cáo tài chính
- Trích xuất tin tức thị trường

**Lưu ý**: Phần này chỉ được xây dựng để kiểm thử, không liên quan đến sản phẩm hiện tại

## Mục lục

1. [Cài đặt và thiếp lập](#scrollTo=BJYQtG0B4-sq)
2. [Danh sách mã chứng khoán](#scrollTo=rdlN42Hl4-st)
3. [Dữ liệu quá khứ và thời gian thực](#scrollTo=fdN7Sobv4-sw)
4. [Dữ liệu tin tức thị trường](#scrollTo=sz8ZmBjF4-sv)

# 1. Cài đặt và thiếp lập
- Cài đặt và import thư viện
- Call API key
- Đăng ký user

In [2]:
import sys
sys.path.append('..') # Nhảy lên 1 cấp ra root

from src.get_news import validate_domains
from src.get_url import fetch_all_raw_links
from src.save_data import save_data
from src.get_data import load_data
import pandas as pd
from vnstock import *
import os

In [4]:
try:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("VNSTOCK_API_KEY") or os.getenv("API_KEY")
    if api_key:
        print("Đã đọc API key từ .env:", api_key[:10] + "...")
    else:
        print("Không tìm thấy API key trong .env (VNSTOCK_API_KEY/API_KEY).")
except Exception:
    print("Cần cài python-dotenv hoặc kiểm tra file .env.")
    api_key = None

Đã đọc API key từ .env: vnstock_73...


In [5]:
if api_key:
    register_user(api_key)
else:
    print("Không có API key, giới hạn 20 lần gọi API mỗi phút")

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***bb89
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)


# 2. Danh sách mã chứng khoán (niêm yết)
- Nguồn:
    - KBS (đang sử dụng)
    - VCI

In [6]:
print("Danh sách các mã chứng khoán (niêm yết):")
listing = Listing(source='KBS')
listing.all_symbols().head(10)

Danh sách các mã chứng khoán (niêm yết):


,symbol,organ_name
0,DPP,CTCP Dược Đồng Nai
1,SDA,CTCP Simco Sông Đà
2,CLH,CTCP Xi măng La Hiên VVMI
3,DBT,CTCP Dược phẩm Bến Tre
4,DND,CTCP Đầu tư Xây dựng và Vật liệu Đồng Nai
5,CCR,CTCP Cảng Cam Ranh
6,NOS,CTCP Vận tải biển và Thương mại Phương Đông
7,DPH,CTCP Dược phẩm Hải Phòng
8,DSN,CTCP Công viên nước Đầm Sen
9,WSS,CTCP Chứng khoán Phố Wall


In [7]:
# Danh sách các mã chứng khoán
len(listing.all_symbols()['symbol'].tolist())

1539

# 3. Dữ liệu quá khứ và thời gian thực

`VCI` là source mạnh nhất nên sẽ được sử dụng chính trong data

In [8]:
source = 'VCI'
quote = Quote(symbol='ACB', source=source)

Cổ phiếu từ hôm nay tới 90 ngày trước

In [9]:
quote.history(length='90', interval='d').head(10)

,time,open,high,low,close,volume
0,2026-01-12,24.65,25.50,24.65,25.50,22302500
1,2026-01-13,25.70,25.80,24.90,24.90,24138300
2,2026-01-14,25.00,25.15,24.50,24.65,19456100
3,2026-01-15,24.85,24.90,24.55,24.90,14169900
4,2026-01-16,25.00,25.10,24.65,24.85,10941400
5,2026-01-19,24.90,25.15,24.75,25.10,10927600
6,2026-01-20,25.10,25.30,25.00,25.05,18384900
7,2026-01-21,25.00,25.15,24.75,24.85,12924400
8,2026-01-22,24.90,25.05,24.80,24.85,10084800
9,2026-01-23,25.00,25.10,24.90,25.05,11781000


Cổ phiếu theo một khoảng thời gian

In [10]:
quote.history(start='2024-01-03', end='2024-01-22', interval='d').head(10)

,time,open,high,low,close,volume
0,2023-12-29,16.77,16.91,16.70,16.77,9202991
1,2024-01-02,16.81,17.37,16.81,17.16,13896933
2,2024-01-03,17.19,17.55,17.02,17.55,9817807
3,2024-01-04,17.69,18.00,17.62,17.76,23605373
4,2024-01-05,17.76,17.86,17.58,17.86,9282598
5,2024-01-08,18.04,18.07,17.69,17.79,12398885
6,2024-01-09,17.72,17.79,17.51,17.55,15455964
7,2024-01-10,17.69,17.90,17.55,17.72,17610661
8,2024-01-11,17.83,18.07,17.65,17.72,9941849
9,2024-01-12,17.65,18.39,17.58,18.11,25574880


Cổ phiếu cập nhật giá mới theo thời gian thực

In [12]:
# Dữ liệu khớp lệnh trong ngày giao dịch realtime hoặc ngày gần nhất (ngoài giờ giao dịch)
quote.intraday(symbol='GVT', page_size=10_000, show_log=False)

,time,price,volume,match_type,id
0,2026-04-21 09:15:00+07:00,24.00,64600,ATO,465927305
1,2026-04-21 09:15:00+07:00,24.00,200,Sell,465927721
2,2026-04-21 09:15:01+07:00,24.00,900,Sell,465927808
3,2026-04-21 09:15:01+07:00,24.00,100,Sell,465927806
4,2026-04-21 09:15:03+07:00,24.00,400,Sell,465927857
...,...,...,...,...,...
3113,2026-04-21 14:29:59+07:00,23.80,100,Buy,466499240
3114,2026-04-21 14:29:59+07:00,23.80,1000,Buy,466499241
3115,2026-04-21 14:29:59+07:00,23.80,1000,Buy,466499242
3116,2026-04-21 14:29:59+07:00,23.80,400,Buy,466499243


# 4. Dữ liệu thị trường

Source `VCI` không thể sử dụng được do một vài lỗi không rõ. Ngoài ra mỗi mã cổ phiếu sẽ cho một thông tin tin tức khác nhau

In [ ]:
source = 'KBS'

In [ ]:
company = Company(symbol='ACB', source=source)
company.news()

,head,article_id,title,publish_time,url
0,Chiến sự Trung Đông leo thang trong tháng 3 đã...,1423566,Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?,2026-04-06T20:02:00,/2026/04/co-phieu-ngan-hang-nao-thoat-lua-trun...


Ngoài ra có một cách lấy thông tin bài báo từ thư viện `newspaper3k`, ở dưới đã tạo một tool để lấy toàn bộ url thô từ mã

In [ ]:
from newspaper import Article

# Domain đúng cho cấu trúc link này là vietstock.vn
full_url = "https://vietstock.vn/2026/04/co-phieu-ngan-hang-nao-thoat-lua-trung-dong-830-1423566.htm"

article = Article(full_url, language='vi')

try:
    article.download()
    article.parse()
    
    print(f"Tiêu đề: {article.title}")
    print("-" * 30)
    print(article.text) # Toàn bộ nội dung bài báo để nạp vào Vector DB
    
except Exception as e:
    print(f"Vẫn lỗi: {e}")

Tiêu đề: Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?
------------------------------
Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?

Chiến sự Trung Đông leo thang trong tháng 3 đã “phả nhiệt” lên thị trường chứng khoán Việt Nam. Tác động khó lường khiến dòng tiền trở nên thận trọng hơn, kéo theo áp lực điều chỉnh ở nhiều nhóm cổ phiếu, trong đó có cả nhóm ngân hàng, vốn được xem là trụ đỡ của thị trường.

Diễn biến chỉ số ngân hàng và VN-Index từ đầu năm 2025 đến đầu tháng 4/2026 Nguồn: VietstockFinance

Khép lại tháng 3 đầy biến động, VN-Index dừng ở mức 1,674.49 điểm, giảm gần 206 điểm chỉ trong một tháng, tương ứng mức sụt giảm khoảng 11%. Trong khi đó, chỉ số ngành ngân hàng cũng mất hơn 9%, lùi về 956.82 điểm tính đến ngày 31/3, cho thấy nhóm này vẫn chịu ảnh hưởng đáng kể từ bối cảnh chung của thị trường.

“Lửa” Trung Đông “thiêu rụi” hơn 250 ngàn tỷ vốn hóa

Nguồn: VietstockFinance

Bất ổn địa chính trị gia tăng làm dấy lên lo ngại về áp lực lạm phát quay trở lại, qua đó đẩy c

Phần này là tool để lấy toàn bộ `url` từ các mã cổ phiếu, file .csv đã tồn tại trong thư mục data nên không cần phải chạy lại lần nữa

In [ ]:
# Chạy và lưu lại file thô
links = listing.all_symbols()['symbol'].tolist() # Lấy tất cả mã niêm yết

# Lấy toàn bộ link thô (có thể mất thời gian)
print(f"Lưu ý: toàn bộ link đã được lưu ở data, nếu bạn đã chạy cell này trước đó. Việc lấy toàn bộ link thô cho {len(links)} mã chứng khoán có thể mất thời gian.")
accept = input(f"Bạn có muốn lấy toàn bộ link thô cho {len(links)} mã chứng khoán không? (y/n): ")
if accept.lower() == 'y':
    df_raw = fetch_all_raw_links(links)
    display(df_raw)
    save_data(df_raw, 'raw_links_backup.csv')
else:
    print("Đã hủy việc lấy toàn bộ link thô.")

Lưu ý: toàn bộ link đã được lưu ở data, nếu bạn đã chạy cell này trước đó. Việc lấy toàn bộ link thô cho 1540 mã chứng khoán có thể mất thời gian.


Đoạn mã dưới đây chỉ là để kiểm tra xem mọi url thô có cùng đuôi`.htm` không, nếu có thì mặc định tên miền là từ vietstock

In [ ]:
url_raw = load_data('raw_links_backup.csv')
display(url_raw.head(10))

,symbol,title,url,publish_time
0,TCB,TCB: Quyết định của HĐQT về việc phê duyệt tha...,/2026/04/tcb-quyet-dinh-cua-hdqt-ve-viec-phe-d...,2026-04-06T15:15:46
1,VGI,VGI: Thông báo ngày đăng ký cuối cùng để thực ...,/2026/03/vgi-thong-bao-ngay-dang-ky-cuoi-cung-...,2026-03-09T12:10:00
2,AGM,AGM: Thông báo thay đổi Người có liên quan của...,/2026/03/agm-thong-bao-thay-doi-nguoi-co-lien-...,2026-03-25T17:19:00
3,DMS,DMS: Ông Hoàng Nam Long được bầu giữ chức Chủ ...,/2025/11/dms-ong-hoang-nam-long-duoc-bau-giu-c...,2025-11-03T16:40:00
4,BKC,BKC: Báo cáo tài chính quý 4/2025 (công ty mẹ),/2026/02/bkc-bao-cao-tai-chinh-quy-4-2025-cong...,2026-02-02T14:06:00
5,VGS,VGS: Thông báo về ngày đăng ký cuối cùng để th...,/2026/03/vgs-thong-bao-ve-ngay-dang-ky-cuoi-cu...,2026-03-04T10:41:00
6,POB,POB: Tài liệu họp Đại hội đồng cổ đông,/2025/11/pob-tai-lieu-hop-dai-hoi-dong-co-dong...,2025-11-26T09:09:00
7,VNG,VNG: Thông báo về ngày đăng ký cuối cùng tham ...,/2026/03/vng-thong-bao-ve-ngay-dang-ky-cuoi-cu...,2026-03-18T13:56:36
8,PVL,Nợ vay doanh nghiệp bất động sản trên sàn lên ...,/2026/02/no-vay-doanh-nghiep-bat-dong-san-tren...,2026-02-24T10:02:00
9,ACE,ACE: Nghị quyết Hội đồng quản trị,/2025/12/ace-nghi-quyet-hoi-dong-quan-tri-737-...,2025-12-30T14:52:00


In [ ]:
# Kiểm tra các link có domain không hợp lệ (không chứa 'htm')
for symbol, url in url_raw[['symbol', 'url']].values:
    if 'htm' not in url:
        print(f"Symbol: {symbol}, URL: {url}")

Vì là không có đuôi khác nên tên miền sẽ cố định là `https://vietstock.vn}`

In [ ]:
import pandas as pd

def attach_vietstock_domain(df_raw):
    """
    Ghép domain Vietstock vào các link tương đối và trả về DataFrame sạch.
    """
    # Tạo bản sao để không ảnh hưởng đến df gốc
    df = df_raw.copy()
    
    # Domain cố định
    domain = "https://vietstock.vn"
    
    # Ghép domain (đảm bảo url bắt đầu bằng /)
    df['valid_url'] = df['url'].apply(lambda x: f"{domain}{x}" if x.startswith('/') else f"{domain}/{x}")
    
    # Chỉ giữ lại các cột cần thiết cho RAG hoặc bước tiếp theo
    return df[['symbol', 'valid_url', 'title', 'publish_time']]

# Cách sử dụng:
df_final = attach_vietstock_domain(url_raw)
df_final.head(10)

,symbol,valid_url,title,publish_time
0,TCB,https://vietstock.vn/2026/04/tcb-quyet-dinh-cu...,TCB: Quyết định của HĐQT về việc phê duyệt tha...,2026-04-06T15:15:46
1,VGI,https://vietstock.vn/2026/03/vgi-thong-bao-nga...,VGI: Thông báo ngày đăng ký cuối cùng để thực ...,2026-03-09T12:10:00
2,AGM,https://vietstock.vn/2026/03/agm-thong-bao-tha...,AGM: Thông báo thay đổi Người có liên quan của...,2026-03-25T17:19:00
3,DMS,https://vietstock.vn/2025/11/dms-ong-hoang-nam...,DMS: Ông Hoàng Nam Long được bầu giữ chức Chủ ...,2025-11-03T16:40:00
4,BKC,https://vietstock.vn/2026/02/bkc-bao-cao-tai-c...,BKC: Báo cáo tài chính quý 4/2025 (công ty mẹ),2026-02-02T14:06:00
5,VGS,https://vietstock.vn/2026/03/vgs-thong-bao-ve-...,VGS: Thông báo về ngày đăng ký cuối cùng để th...,2026-03-04T10:41:00
6,POB,https://vietstock.vn/2025/11/pob-tai-lieu-hop-...,POB: Tài liệu họp Đại hội đồng cổ đông,2025-11-26T09:09:00
7,VNG,https://vietstock.vn/2026/03/vng-thong-bao-ve-...,VNG: Thông báo về ngày đăng ký cuối cùng tham ...,2026-03-18T13:56:36
8,PVL,https://vietstock.vn/2026/02/no-vay-doanh-nghi...,Nợ vay doanh nghiệp bất động sản trên sàn lên ...,2026-02-24T10:02:00
9,ACE,https://vietstock.vn/2025/12/ace-nghi-quyet-ho...,ACE: Nghị quyết Hội đồng quản trị,2025-12-30T14:52:00


In [ ]:
# Lưu lại file đã xử lý
save_data(df_final, 'processed_links.csv')

✅ Data saved to: c:\Users\YOGA\Desktop\06_SQL_v2\project\stock-tracker\data\processed_links.csv


Cuối cùng là thử nghiệm việc lấy báo qua link bất kỳ

In [ ]:
from newspaper import Article, Config
import pandas as pd

def extract_news_to_schema(symbol, df_verified, limit=5):
    """
    Đầu vào: symbol (mã CK), df_verified (DataFrame chứa cột symbol và valid_url)
    Đầu ra: List các dict theo khung chuẩn để insert vào Database
    """
    # 1. Lọc lấy các link thuộc về symbol này
    links_to_crawl = df_verified[df_verified['symbol'] == symbol].head(limit)
    
    if links_to_crawl.empty:
        print(f"Không có link nào cho mã {symbol}")
        return []

    # 2. Cấu hình newspaper3k (Fake User-Agent để tránh bị Vietstock chặn)
    config = Config()
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    config.request_timeout = 15

    structured_data = []

    for _, row in links_to_crawl.iterrows():
        url = row['valid_url']
        try:
            article = Article(url, config=config, language='vi')
            article.download()
            article.parse()
            
            # 3. Đưa vào khung chuẩn (Schema)
            news_item = {
                "symbol": symbol,
                "title": article.title,
                "content": article.text,  # Toàn bộ nội dung để vector hóa
                "url": url,
                "publish_date": row.get('publish_time', None), # Lấy từ df gốc của vnstock
                "author": ", ".join(article.authors) if article.authors else "Unknown",
                "source": "Vietstock"
            }
            
            structured_data.append(news_item)
            print(f"✅ Đã trích xuất: {article.title[:50]}...")
            
        except Exception as e:
            print(f"❌ Lỗi khi cào link {url}: {e}")
            
    return structured_data

extract_news_to_schema('ACB', df_final)

✅ Đã trích xuất: Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?...


[{'symbol': 'ACB',
  'title': 'Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?',
  'content': 'Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?\n\nChiến sự Trung Đông leo thang trong tháng 3 đã “phả nhiệt” lên thị trường chứng khoán Việt Nam. Tác động khó lường khiến dòng tiền trở nên thận trọng hơn, kéo theo áp lực điều chỉnh ở nhiều nhóm cổ phiếu, trong đó có cả nhóm ngân hàng, vốn được xem là trụ đỡ của thị trường.\n\nDiễn biến chỉ số ngân hàng và VN-Index từ đầu năm 2025 đến đầu tháng 4/2026 Nguồn: VietstockFinance\n\nKhép lại tháng 3 đầy biến động, VN-Index dừng ở mức 1,674.49 điểm, giảm gần 206 điểm chỉ trong một tháng, tương ứng mức sụt giảm khoảng 11%. Trong khi đó, chỉ số ngành ngân hàng cũng mất hơn 9%, lùi về 956.82 điểm tính đến ngày 31/3, cho thấy nhóm này vẫn chịu ảnh hưởng đáng kể từ bối cảnh chung của thị trường.\n\n“Lửa” Trung Đông “thiêu rụi” hơn 250 ngàn tỷ vốn hóa\n\nNguồn: VietstockFinance\n\nBất ổn địa chính trị gia tăng làm dấy lên lo ngại về áp lực lạm phát quay tr

In [ ]:
# Giả sử bạn muốn lấy tin cho nhóm VN30 trước
symbols_to_run = ['ACB', 'FPT', 'VNM', 'MSN', 'VIC']
all_news_payload = []

for s in symbols_to_run:
    # Gọi hàm trích xuất
    data = extract_news_to_schema(s, df_final, limit=3)
    all_news_payload.extend(data)

print(f"Tổng số bài báo đã trích xuất: {len(all_news_payload)}")
# Sau đó dùng all_news_payload để insert vào DB
# insert_to_cassandra(all_news_payload)

✅ Đã trích xuất: Cổ phiếu ngân hàng nào “thoát lửa” Trung Đông?...
✅ Đã trích xuất: FPT: Báo cáo kết quả giao dịch cổ phiếu của tổ chứ...
✅ Đã trích xuất: Vị thế tiền mặt ròng là gì và ai đang đứng đầu trê...
✅ Đã trích xuất: Mảng tiêu dùng – bán lẻ khởi sắc, Masan kỳ vọng độ...
✅ Đã trích xuất: Vingroup tăng tốc tại Ấn Độ, nhắm hệ sinh thái 6.5...
Tổng số bài báo đã trích xuất: 5
